# 2 · Von der MIDI-Datei zur neuen Melodie

[![In Colab öffnen](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sem-16161d/GreasyPidginTutorial/blob/main/DE/jupyter/markov-book-de/02_von_midi_zur_melodie.ipynb)

Jetzt sind die Symbole **Noten**. Der Plan:

1. eine Sonate von Scarlatti wählen (Standard: K. 555)
2. GreasyPidgin liest sie ein
3. lernen, welche Note auf welche zu folgen pflegt
4. eine neue Melodie erzeugen
5. sie in eine MIDI-Datei schreiben, die du herunterladen kannst

In [ ]:
# Einrichtung -- läuft von selbst (beim ersten Mal etwa 20 Sekunden)
import importlib.util, pathlib, subprocess, sys, urllib.request

GP_REF        = "main"   # welche Version von GreasyPidgin verwendet wird (Git-Branch oder -Tag)
TUTORIAL_REF  = "main"   # welche Version der Hilfsdatei dieses Tutorials verwendet wird
GP_REPO       = "sem-16161d/GreasyPidgin"
TUTORIAL_REPO = "sem-16161d/GreasyPidginTutorial"

if importlib.util.find_spec("GreasyPidgin") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check",
                    f"git+https://github.com/{GP_REPO}.git@{GP_REF}"], check=True)

if not pathlib.Path("workshop_helpers.py").exists():
    urllib.request.urlretrieve(
        f"https://raw.githubusercontent.com/{TUTORIAL_REPO}/{TUTORIAL_REF}/DE/jupyter/markov-book-de/workshop_helpers.py",
        "workshop_helpers.py")

import random
import matplotlib.pyplot as plt
from GreasyPidgin import MarkovChain, MarkovRule, MarkovRules
import workshop_helpers as wh

wh.set_language("de")    # deutsche Meldungen, Beschriftungen und Notennamen
print("Bereit!")

## Schritt 1: eine Sonate wählen

Als Trainingsmaterial dient eine der 555 Cembalosonaten von Domenico Scarlatti (Zählung nach Kirkpatrick, K. 1 bis K. 555).
Standardmäßig ist es die letzte: **K. 555**.

**Ändere die Zahl in der Zelle unten** (1 bis 555) und führe die Zelle aus, um mit einer anderen Sonate zu trainieren.
Die Sonaten haben zwei Hände; beginnen mehrere Noten gleichzeitig, bleibt nur die höchste erhalten.
Eine eigene MIDI-Datei? Siehe „Probier es aus" ganz unten. (MIDI-Dateien: © Sankey, laut Dateiname.)

In [ ]:
SONATA = 555      # <- ändere diese Zahl (1 bis 555), um mit einer anderen Sonate zu trainieren

midi_path = wh.get_scarlatti(SONATA, TUTORIAL_REPO, TUTORIAL_REF)

## Schritt 2: mit GreasyPidgin einlesen

Im Hintergrund verwandelt `Gesture.fromMidiFile` jede MIDI-Note in ein **Phonon** — das Notenobjekt von
GreasyPidgin — und sammelt sie in einer **Gesture**. Wir holen Tonhöhen und Rhythmen heraus und
bewahren sie in einem kleinen `Melody`-Objekt auf.

In [ ]:
melody = wh.load_melody(midi_path)
print(melody)
wh.plot_melody(melody)
plt.show()

In [ ]:
wh.listen(midi_path)

`melody.pitches` sind MIDI-Notennummern (60 = mittleres C, geschrieben `c4`; `s` heißt Kreuz und `f` heißt b, also ist `cs4` = Cis4 und `bf3` = B3; deutsches H ist `b`), `melody.deltas` sind die Zeiten
zwischen den Notenanfängen in Schlägen (1.0 = eine Viertelnote), `melody.bpm` ist das Tempo.

In [ ]:
print(melody.pitches[:12])
print(melody.deltas[:12])
print(melody.bpm)

## Schritt 3: die Kette lernen

`ORDER` ist das Gedächtnis: wie viele vorherige Noten die nächste bestimmen.

In [ ]:
ORDER = 2

pitch_chain = MarkovChain()
pitch_chain.generateRulesFromList(melody.pitches, order=ORDER)
print(f"{len(pitch_chain.rules)} Regeln gelernt. Die ersten drei:")

for state in pitch_chain.rules.elements[:3]:
    names = " ".join(wh.pitch_name(p) for p in state)
    print(f"  nach  {names:8s} ->", pitch_chain.rules[state])

## Schritt 4: erzeugen

Wir beginnen die neue Melodie mit den ersten Noten des Originals und lassen die Kette weiterschreiben.

In [ ]:
N_NOTES = 32

start = tuple(melody.pitches[:ORDER])
new_pitches = pitch_chain.getNElements(seed=start, n=N_NOTES)

print(" ".join(wh.pitch_name(p) for p in new_pitches))

## Schritt 5: ein Rhythmus und eine MIDI-Datei

Vorerst borgt sich die neue Melodie den Rhythmus des Originals (und wiederholt ihn so oft wie nötig).
Im nächsten Kapitel lernt die Kette auch den Rhythmus.

Wir bauen eine **Gesture** aus parallelen Listen: eine Liste für die Tonhöhen, eine für die Zeit
zwischen den Einsätzen, eine für die Dauern, dazu eine Dynamik und ein Tempo.

In [ ]:
from GreasyPidgin import Gesture

rhythm = [melody.deltas[i % len(melody.deltas)] for i in range(len(new_pitches))]

gesture = Gesture.fromLists(
    size        = len(new_pitches),
    startOffset = 0,
    startDeltas = [0] + rhythm[:-1],           # Zeit vor dem Einsatz jeder Note
    durations   = [r * 0.9 for r in rhythm],   # leicht abgesetzt
    pitches     = new_pitches,
    dynamics    = 80,                          # MIDI-Velocity
    timeUnit    = "b",                         # alle Zeiten in Schlägen
    bpm         = melody.bpm,
)

gesture.toMidi("new_melody.mid", bpm=melody.bpm)

In [ ]:
wh.plot_notes(new_pitches, rhythm, title="Die neue Melodie", color="C1")
plt.show()
wh.listen("new_melody.mid")

Gefällt sie dir? Dann speichere die MIDI-Datei auf deinem Computer:

In [ ]:
wh.download("new_melody.mid")

## Ein anderer Ansatz: Intervalle statt Noten

Bisher hat die Kette **absolute Noten** gelernt: „nach `c5` kommt `ef5`". Sie spielt nur Noten, die im Original vorkamen,
und immer in derselben Lage. Jetzt lassen wir sie stattdessen **Intervalle** lernen: den Abstand zwischen zwei
aufeinanderfolgenden Noten in Halbtönen (`+2` = große Sekunde aufwärts, `-3` = kleine Terz abwärts).
Ein Intervall wie „zwei Halbtöne hinauf" kommt an vielen Stellen vor, egal auf welcher Note. Die Kette lernt also die
*Gestalt* der Melodie, nicht ihre Tonhöhen.
Intervalle sind überall gleich, daher kann die Kette aus **mehreren Sonaten gleichzeitig** lernen: Trag ihre Nummern
unten in `SONATAS` ein (eine einzelne Nummer geht auch). Die Sonaten werden einfach hintereinander gehängt.

In [ ]:
SONATAS = [555, 554, 553]    # <- eine Sonate oder mehrere, z. B. [100] oder [100, 101, 102]

intervals = []
all_pitches = []
for number in SONATAS:
    m = wh.load_melody(wh.get_scarlatti(number, TUTORIAL_REPO, TUTORIAL_REF))
    intervals += [b - a for a, b in zip(m.pitches, m.pitches[1:])]
    all_pitches += m.pitches
print(intervals[:12])      # Halbtonschritte zwischen aufeinanderfolgenden Noten

interval_chain = MarkovChain()
interval_chain.generateRulesFromList(intervals, order=ORDER)
print(f"{len(interval_chain.rules)} Regeln aus {len(SONATAS)} Sonate(n) gelernt "
      f"(die Noten-Kette allein auf K. {SONATA} hatte {len(pitch_chain.rules)}).")

Zum Erzeugen starten wir auf der ersten Note des Originals und addieren die gelernten Intervalle nacheinander.
Läuft die Melodie aus dem Tonumfang des Originals heraus, setzen wir sie um Oktaven zurück.

In [ ]:
low, high = min(all_pitches), max(all_pitches)      # Tonumfang aller Trainingssonaten

new_intervals = interval_chain.getNElements(seed=tuple(intervals[:ORDER]), n=N_NOTES)

pitch = melody.pitches[0]            # Startnote: die erste Note des Originals
interval_pitches = [pitch]
for step in new_intervals[:N_NOTES - 1]:
    pitch += step
    while pitch > high:              # im Tonumfang des Originals bleiben
        pitch -= 12
    while pitch < low:
        pitch += 12
    interval_pitches.append(pitch)

print(" ".join(wh.pitch_name(p) for p in interval_pitches))

In [ ]:
wh.render_midi(interval_pitches, rhythm, melody.bpm, "new_melody_intervals.mid")
wh.plot_notes(interval_pitches, rhythm, title="Die neue Melodie (Intervalle)", color="C2")
plt.show()
wh.listen("new_melody_intervals.mid")

Hör beide Melodien nacheinander. Die Noten-Kette bleibt bei den Tönen und Wendungen des Originals. Die Intervall-Kette
setzt dessen Bewegungen an beliebigen Stellen neu zusammen und kann dabei Töne treffen, die im Original nie vorkamen.

In [ ]:
wh.download("new_melody_intervals.mid")

## Akkorde: alle Noten behalten

Bisher wurde jeder Akkord auf seine höchste Note reduziert, und der Rhythmus stammte vom Original.
Jetzt hört die Kette **alles**. Ein Akkord sind einfach Noten, die zur gleichen Zeit beginnen, von der tiefsten aufwärts
geschrieben, und die Kette behandelt sie wie jede andere Notenfolge.

Jede Note ist ein Symbol aus zwei Zahlen: `(interval, gap)`.

- `interval` ist der Abstand in Halbtönen zur vorigen Note, wie bisher.
- `gap` ist die Zeit seit dem Beginn der vorigen Note, in Schlägen. **`gap = 0` ist erlaubt:** Die Note beginnt zusammen
  mit der vorigen, beide bilden also einen Akkord.

Die Kette lernt den Rhythmus zusammen mit den Intervallen.

In [ ]:
import io, contextlib

chord_symbols = []           # ein Symbol pro Note: (interval, gap)
used_pitches = []

for number in SONATAS:
    with contextlib.redirect_stdout(io.StringIO()):     # Gesture.fromMidiFile gibt jede Note aus
        g = Gesture.fromMidiFile(wh.get_scarlatti(number, TUTORIAL_REPO, TUTORIAL_REF))
    # alle Noten, auch Akkorde, als (Start in Schlägen, Tonhöhe); im Akkord von tief nach hoch
    notes = sorted({(round(p.startTimeBeats, 3), round(q)) for p in g.children for q in p.allPitches})
    for (t0, a), (t1, b) in zip(notes, notes[1:]):
        chord_symbols.append((b - a, round(t1 - t0, 1)))     # (interval, gap); gap 0 = gleichzeitig mit der Note davor
    used_pitches += [q for _, q in notes]

print(chord_symbols[:8])

chord_chain = MarkovChain()
chord_chain.generateRulesFromList(chord_symbols, order=ORDER)
in_chords = sum(1 for _, gap in chord_symbols if gap == 0)
print(f"{len(chord_symbols)} Noten, davon {in_chords} in Akkorden. {len(chord_chain.rules)} Regeln gelernt.")

Zum Erzeugen startet die Kette in einem zufälligen gelernten Zustand und addiert Intervall um Intervall; sie wird wie bisher
um Oktaven in den Tonumfang zurückgesetzt. Noten mit `gap = 0` erklingen zusammen.

In [ ]:
N_NOTES = 200
low, high = min(used_pitches), max(used_pitches)

seed = random.choice(chord_chain.rules.elements)           # ein zufälliger Startzustand
new_symbols = chord_chain.getNElements(seed=seed, n=N_NOTES)

pitch = melody.pitches[0]
chord_pitches, gaps = [pitch], [0]
for step, gap in new_symbols[:N_NOTES - 1]:
    pitch += step
    while pitch > high:                       # im Tonumfang der Originale bleiben
        pitch -= 12
    while pitch < low:
        pitch += 12
    chord_pitches.append(pitch)
    gaps.append(gap)

groups = []                                   # Akkorde als c4+e4+g4 ausgeben
for p, gap in zip(chord_pitches, gaps):
    if gap == 0 and groups:
        groups[-1].append(wh.pitch_name(p))
    else:
        groups.append([wh.pitch_name(p)])
print("  ".join("+".join(g) for g in groups))

In [ ]:
# jede Note klingt, bis der nächste Akkord beginnt
durations = [next((g for g in gaps[i + 1:] if g > 0), 1.0) * 0.9 for i in range(len(gaps))]

chord_gesture = Gesture.fromLists(
    size        = len(chord_pitches),
    startOffset = 0,
    startDeltas = gaps,                       # 0 = gleichzeitig mit der Note davor: ein Akkord
    durations   = durations,
    pitches     = chord_pitches,
    dynamics    = 80,
    timeUnit    = "b",
    bpm         = melody.bpm,
)
chord_gesture.toMidi("new_melody_chords.mid", bpm=melody.bpm)

wh.plot_notes(chord_pitches, gaps[1:] + [1.0], durations,    # plot_notes will die Zeit NACH jeder Note
              title="Die neue Melodie (mit Akkorden)", color="C3")
plt.show()
wh.listen("new_melody_chords.mid")

Akkorde entstehen überall dort, wo die Kette ein `gap` von `0` zieht. Auch der Rhythmus kommt jetzt von der Kette, nicht vom Original.

In [ ]:
wh.download("new_melody_chords.mid")

## Höhere Ordnung: ein längeres Gedächtnis

Bisher hat die Kette auf die letzten `ORDER` Noten geschaut. Mit einer **höheren Ordnung** erinnert sie sich an ein längeres
Stück Musik, die Melodie folgt dem Original also enger. Der Preis: Je höher die Ordnung, desto weniger Möglichkeiten bleiben,
und irgendwann kann die Kette das Original nur noch wiederholen. Die Tabelle zeigt für jede Ordnung, wie viel einer neuen Melodie
vom Original kopiert ist (Anteil an Gruppen aus 4 Noten). Danach erzeugt die Zelle mit `HIGH_ORDER`.

In [ ]:
HIGH_ORDER = 3          # Gedächtnis: wie viele vorige Noten die nächste bestimmen
N_NOTES = 200

for order in (1, 2, 3, 4):             # wie viel vom Original kopiert jede Ordnung?
    chain = MarkovChain()
    chain.generateRulesFromList(chord_symbols, order=order)
    new = chain.getNElements(seed=random.choice(chain.rules.elements), n=N_NOTES)
    print(f"Ordnung {order}: {len(chain.rules):5d} Regeln, {wh.copy_ratio(chord_symbols, new):4.0%} kopiert")

high_chain = MarkovChain()
high_chain.generateRulesFromList(chord_symbols, order=HIGH_ORDER)
high_symbols = high_chain.getNElements(seed=random.choice(high_chain.rules.elements), n=N_NOTES)

In [ ]:
def render_chords(pitches, gaps, durations, path, title, color):
    gesture = Gesture.fromLists(
        size=len(pitches), startOffset=0, startDeltas=gaps, durations=durations,
        pitches=pitches, dynamics=80, timeUnit="b", bpm=melody.bpm,
    )
    gesture.toMidi(path, bpm=melody.bpm)
    wh.plot_notes(pitches, gaps[1:] + [1.0], durations, title=title, color=color)
    plt.show()
    wh.listen(path)


pitch = melody.pitches[0]
high_pitches, gaps = [pitch], [0]
for step, gap in high_symbols[:N_NOTES - 1]:
    pitch += step
    while pitch > high:                       # im Tonumfang der Originale bleiben
        pitch -= 12
    while pitch < low:
        pitch += 12
    high_pitches.append(pitch)
    gaps.append(gap)

durations = [next((g for g in gaps[i + 1:] if g > 0), 1.0) * 0.9 for i in range(len(gaps))]
render_chords(high_pitches, gaps, durations, "new_melody_high_order.mid", "Die neue Melodie (höhere Ordnung)", "C4")

In [ ]:
wh.download("new_melody_high_order.mid")

## Unabhängige Ketten: Tonhöhe, Abstand und Dauer getrennt

Bisher trug ein Symbol alles: `(interval, gap)`. Jetzt lernen drei **getrennte Ketten** die Intervalle, die Abstände (Zeit
zwischen den Anfängen, `0` = Akkord) und die Dauern, jede mit dem Gedächtnis `HIGH_ORDER`. Sie wissen nichts voneinander. Tonhöhe und Rhythmus
werden unabhängig gemischt, sodass Kombinationen entstehen, die im Original nie vorkamen.

In [ ]:
interval_list, gap_list, duration_list = [], [], []

for number in SONATAS:
    with contextlib.redirect_stdout(io.StringIO()):
        g = Gesture.fromMidiFile(wh.get_scarlatti(number, TUTORIAL_REPO, TUTORIAL_REF))
    # alle Noten als (Start in Schlägen, Tonhöhe, Dauer); im Akkord von tief nach hoch
    notes = sorted({(round(p.startTimeBeats, 3), round(q), max(0.1, round(p.durationBeats, 1)))
                    for p in g.children for q in p.allPitches})
    for (t0, a, _), (t1, b, d) in zip(notes, notes[1:]):
        interval_list.append(b - a)
        gap_list.append(round(t1 - t0, 1))
        duration_list.append(d)

interval_chain3, gap_chain, duration_chain = MarkovChain(), MarkovChain(), MarkovChain()
interval_chain3.generateRulesFromList(interval_list, order=HIGH_ORDER)
gap_chain.generateRulesFromList(gap_list, order=HIGH_ORDER)
duration_chain.generateRulesFromList(duration_list, order=HIGH_ORDER)
print(f"Regeln gelernt: Intervalle {len(interval_chain3.rules)}, Abstände {len(gap_chain.rules)}, Dauern {len(duration_chain.rules)}.")

In [ ]:
N_NOTES = 200

def draw(chain):                              # n Elemente aus einem zufälligen Startzustand
    return chain.getNElements(seed=random.choice(chain.rules.elements), n=N_NOTES)

new_intervals = draw(interval_chain3)
new_gaps      = draw(gap_chain)
new_durations = draw(duration_chain)

pitch = melody.pitches[0]
indep_pitches, gaps = [pitch], [0]
for step, gap in zip(new_intervals[:N_NOTES - 1], new_gaps):
    pitch += step
    while pitch > high:
        pitch -= 12
    while pitch < low:
        pitch += 12
    indep_pitches.append(pitch)
    gaps.append(gap)

render_chords(indep_pitches, gaps, new_durations[:N_NOTES], "new_melody_independent.mid", "Die neue Melodie (unabhängige Ketten)", "C5")

In [ ]:
wh.download("new_melody_independent.mid")

## Nochmal, nochmal!

Geh zurück zu **Schritt 4** und führe die Zellen darunter erneut aus: jeder Durchlauf ergibt eine neue Melodie.

**Probier es aus:**

- Ändere `ORDER` in Schritt 3 auf `1` oder `3` (und führe Schritt 3 und 4 erneut aus). Klingt die neue Melodie
  noch wie das Original?
- Mach sie länger (`N_NOTES = 100`).
- Starte die Intervall-Melodie auf einer anderen Note: setze `pitch = 60` statt `pitch = melody.pitches[0]`.
  Sie hat dieselbe Gestalt, nur versetzt.
- Trainiere mit mehr Sonaten: trag mehr Nummern in `SONATAS` ein, z. B. `SONATAS = [100, 101, 102, 103]`, und führe die
  Intervall- und Akkord-Zellen erneut aus.
- Akkorde: ändere `ORDER` und vergleiche, wie viele Akkorde entstehen. Oder starte die Akkord-Melodie auf einer anderen Note: setze in der Akkord-Zelle `pitch = 48` statt `pitch = melody.pitches[0]`.
- Ändere `HIGH_ORDER` auf `1`, `2` oder `6`. Was passiert mit dem kopierten Anteil in der Tabelle und mit den unabhängigen Ketten weiter unten?
- Wähle eine andere Sonate: ändere `SONATA` in Schritt 1 und führe alle Zellen ab Schritt 1 erneut aus.
- Lade eine eigene MIDI-Datei hoch: ersetze die Zeile `midi_path = ...` in Schritt 1 durch `midi_path = wh.choose_midi()`.
- Ändere die `0.9` in Schritt 5 auf `1.0` (legato) oder `0.4` (staccato).